# MACD Histogram Acceleration on SPY
## Strategy Brief
This strategy involves using the acceleration of the MACD histogram as a momentum indicator to predict potential price movements in the SPY ETF. The MACD histogram acceleration is calculated as the difference between consecutive MACD histogram values. A positive acceleration suggests increasing bullish momentum, while a negative acceleration indicates increasing bearish momentum. The trading logic involves entering a long position when the MACD histogram acceleration turns positive and exiting when it turns negative. This strategy aims to capture short-term momentum shifts, potentially leading to profitable trades.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for our trading strategy, including the lookback period for the MACD calculation and the stock ticker symbol.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuration
TICKER = 'SPY'
START_DATE = '2010-01-01'
END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')
MACD_SHORT_PERIOD = 12
MACD_LONG_PERIOD = 26
MACD_SIGNAL_PERIOD = 9

### PHASE 2 - Data Exploration
In this phase, we download historical price data for SPY from Yahoo Finance, compute the MACD and its histogram, and visualize these indicators overlaid on the price chart.

In [ ]:
def compute_macd(data, short_period, long_period, signal_period):
    short_ema = data['Close'].ewm(span=short_period, adjust=False).mean()
    long_ema = data['Close'].ewm(span=long_period, adjust=False).mean()
    macd = short_ema - long_ema
    signal = macd.ewm(span=signal_period, adjust=False).mean()
    macd_histogram = macd - signal
    return macd, signal, macd_histogram

# Download data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute MACD and histogram
data['MACD'], data['Signal'], data['MACD_Histogram'] = compute_macd(data, MACD_SHORT_PERIOD, MACD_LONG_PERIOD, MACD_SIGNAL_PERIOD)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Price')
plt.plot(data['MACD'], label='MACD', linestyle='--')
plt.plot(data['Signal'], label='Signal', linestyle='--')
plt.bar(data.index, data['MACD_Histogram'], label='MACD Histogram', color='grey', alpha=0.3)
plt.legend()
plt.title('SPY Price and MACD Indicators')
plt.show()

### PHASE 3 - Strategy Engineering
Here, we develop the trading signal based on the acceleration of the MACD histogram. The signal is positive when the acceleration is positive, indicating a buy signal, and negative when the acceleration is negative, indicating a sell signal.

In [ ]:
# Calculate MACD Histogram Acceleration
data['MACD_Histogram_Acceleration'] = data['MACD_Histogram'].diff()

# Generate signals
data['Signal'] = 0

data.loc[data['MACD_Histogram_Acceleration'] > 0, 'Signal'] = 1

data.loc[data['MACD_Histogram_Acceleration'] < 0, 'Signal'] = -1

# Positions
positions = data['Signal']

### PHASE 4 - Coding & Backtesting
We backtest the strategy by shifting the positions to simulate entering trades at the close of the signal day. We then calculate daily returns and plot the equity curve of the strategy.

In [ ]:
# Shift positions to simulate trading at the next day's open
positions = positions.shift(1)

# Calculate daily returns
returns = data['Close'].pct_change()
strategy_returns = positions * returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.plot((1 + returns).cumprod(), label='Buy and Hold Equity Curve', linestyle='--')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We also compare these metrics to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (1 + returns).prod() ** (252 / len(returns)) - 1
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = returns.mean() / downside_returns.std() * np.sqrt(252)
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd = calculate_performance_metrics(strategy_returns)
bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd = calculate_performance_metrics(returns)

# Performance comparison table
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd],
    'Buy and Hold': [bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd]
})
print(performance_df)

### PHASE 6 - Deploy & Monitor
Finally, we create a function to download the last 60 days of data, compute the MACD histogram acceleration, and print today's trading signal.

In [ ]:
def get_latest_signal(ticker, short_period, long_period, signal_period):
    recent_data = yf.download(ticker, period='60d')
    macd, signal, macd_histogram = compute_macd(recent_data, short_period, long_period, signal_period)
    macd_histogram_acceleration = macd_histogram.diff()
    latest_signal = 0
    if macd_histogram_acceleration.iloc[-1] > 0:
        latest_signal = 1
    elif macd_histogram_acceleration.iloc[-1] < 0:
        latest_signal = -1
    print(f"Latest signal for {ticker}: {'Buy' if latest_signal == 1 else 'Sell' if latest_signal == -1 else 'Hold'}")

get_latest_signal(TICKER, MACD_SHORT_PERIOD, MACD_LONG_PERIOD, MACD_SIGNAL_PERIOD)